In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

#dir = "/cluster/projects/bhklab/Users/julian/PMLB"
dir = "/Users/julianguyen/Documents/multimodal-pmlb"
os.chdir(dir)
sys.path.insert(0, dir)
from workflow.notebooks.utils.early_helpers import *

np.random.seed(101)

# set parent folder name for analysis outfiles
OUTDIR = "4-CommonGene1"

In [ ]:
# load in files
atc = pd.read_csv("data/procdata/model_cg1/atc.csv", index_col=0)
rna = pd.read_csv("data/procdata/model_cg1/rna.csv", index_col=0)
cnv = pd.read_csv("data/procdata/model_cg1/cnv.csv", index_col=0)
mut = pd.read_csv("data/procdata/model_cg1/mut.csv", index_col=0)
pheno = pd.read_csv("data/procdata/model_cg1/meta.csv", index_col=0)

# get targest
dt = pheno["doubling_rate"]

# log normalize RNA counts
rna = np.log2(rna + 1)

# scale CNV to -1:1 (currently -2:2)
cnv = cnv/2

# binarize mutations
mut = (mut > 0).astype(int)

# Feature Selection via LASSO Selection Distribution

In [ ]:
run_lasso(atc, dt, "4-CommonGene1/lasso/atac_", repeats=50, features=True)
run_lasso(rna, dt, "4-CommonGene1/lasso/rna_", repeats=50, features=True)
run_lasso(cnv, dt, "4-CommonGene1/lasso/cnv_", repeats=50, features=True)
run_lasso(mut, dt, "4-CommonGene1/lasso/mut_", repeats=50, features=True)

In [ ]:
al = pd.read_csv(f"data/results/data/{OUTDIR}/lasso/atac_lasso_feature_stability.csv", index_col=0)
rl = pd.read_csv(f"data/results/data/{OUTDIR}/lasso/rna_lasso_feature_stability.csv", index_col=0)
cl = pd.read_csv(f"data/results/data/{OUTDIR}/lasso/cnv_lasso_feature_stability.csv", index_col=0)
ml = pd.read_csv(f"data/results/data/{OUTDIR}/lasso/mut_lasso_feature_stability.csv", index_col=0)

al.head()

In [ ]:
def plot_features_cumulative(df, title, ax):

    bins = np.arange(0, 1 + 0.1, 0.1)

    counts, bins, patches = ax.hist(
        df["frequency"],
        bins=bins,
        cumulative=-1,
        edgecolor="black",
        color="#998888",
        alpha=0.85
    )

    for count, left, right in zip(counts, bins[:-1], bins[1:]):
        if count > 0:
            x = (left + right) / 2
            ax.text(x, count + 0.3, f"{int(count)}",
                    ha="center", va="bottom", fontsize=9)

    ax.set_title(title, fontsize=11, weight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    ax.grid(False, axis="x")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(9, 7))

plot_features_cumulative(al, "ATAC", ax=axes[0, 0])
plot_features_cumulative(rl, "RNA",  ax=axes[0, 1])
plot_features_cumulative(cl, "CNV",  ax=axes[1, 0])
plot_features_cumulative(ml, "MUT",  ax=axes[1, 1])

for ax in axes.flat:
    ax.invert_xaxis()
    ax.set_xlim(1, 0)
    ax.set_xticks(np.arange(0, 1.1, 0.1))
    ax.tick_params(axis='x', which='major', length=4)

fig.text(0.5, 0.04, "Frequency threshold", ha="center", fontsize=12)
fig.text(0.04, 0.5, "Number of genes ≥ threshold", va="center", rotation="vertical", fontsize=12)

plt.tight_layout(rect=[0.05, 0.05, 1, 1])
plt.show()

In [ ]:
a_stable = al[al['frequency'] >= 0.3]
print(a_stable.shape[0])

r_stable = rl[rl['frequency'] >= 0.3]
print(r_stable.shape[0])

c_stable = cl[cl['frequency'] >= 0.3]
print(c_stable.shape[0])

m_stable = ml[ml['frequency'] >= 0.3]
print(m_stable.shape[0])

# Unimodal Models
### 1 - With omic-specific unimodal selected features

In [ ]:
# subset each omics for omic-specific stable features
atc_uni_selected = atc.loc[:, atc.columns.intersection(a_stable.index)]
rna_uni_selected = rna.loc[:, rna.columns.intersection(r_stable.index)]
cnv_uni_selected = cnv.loc[:, cnv.columns.intersection(c_stable.index)]
mut_uni_selected = mut.loc[:, atc.columns.intersection(m_stable.index)]

print(atc_uni_selected.shape)
print(rna_uni_selected.shape)
print(cnv_uni_selected.shape)
print(mut_uni_selected.shape)

In [ ]:
SUBDIR = "unimodal-1"

# ATAC
assert (atc_uni_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(atc_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/atac_", repeats=10, preds=True, features=True)
run_random_forest(atc_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/atac_", repeats=10, preds=True, features=True)

# RNA
assert (rna_uni_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(rna_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/rna_", repeats=10, preds=True, features=True)
run_random_forest(rna_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/rna_", repeats=10, preds=True, features=True)

# CNV
assert (cnv_uni_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(cnv_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/cnv_", repeats=10, preds=True, features=True)
run_random_forest(cnv_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/cnv_", repeats=10, preds=True, features=True)

# MUT
assert (mut_uni_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(mut_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/mut_", repeats=10, preds=True, features=True)
run_random_forest(mut_uni_selected, dt, f"{OUTDIR}/{SUBDIR}/mut_", repeats=10, preds=True, features=True)

### 2 - With features selected by >1 omics

In [ ]:
# keep features that are selected by at least two modalities
dfs = [a_stable, r_stable, c_stable, m_stable]
row_counts = Counter()
for df in dfs:
    row_counts.update(df.index)
common_features = {row for row, count in row_counts.items() if count >= 2}
print(f"Number of features in >1 omics: {len(common_features)}\n")

# subset omic matrices for selected features
def subset_omics(stable, omics):
    stable_common = stable[stable.index.isin(common_features)]
    omics = omics.loc[:, omics.columns.intersection(stable_common.index)]
    print(omics.shape)
    return omics

a_selected = subset_omics(a_stable, atc)
r_selected = subset_omics(r_stable, rna)
c_selected = subset_omics(c_stable, cnv)
m_selected = subset_omics(m_stable, mut)

In [ ]:
SUBDIR = "unimodal-2"

# ATAC
assert (a_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(a_selected, dt, f"{OUTDIR}/{SUBDIR}/atac_", repeats=10, preds=True, features=True)
run_random_forest(a_selected, dt, f"{OUTDIR}/{SUBDIR}/atac_", repeats=10, preds=True, features=True)

# RNA
assert (r_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(r_selected, dt, f"{OUTDIR}/{SUBDIR}/rna_", repeats=10, preds=True, features=True)
run_random_forest(r_selected, dt, f"{OUTDIR}/{SUBDIR}/rna_", repeats=10, preds=True, features=True)

# CNV
assert (c_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(c_selected, dt, f"{OUTDIR}/{SUBDIR}/cnv_", repeats=10, preds=True, features=True)
run_random_forest(c_selected, dt, f"{OUTDIR}/{SUBDIR}/cnv_", repeats=10, preds=True, features=True)

# MUT
assert (m_selected.index == dt.index).all(), "sample mismatch"
run_elastic_net(m_selected, dt, f"{OUTDIR}/{SUBDIR}/mut_", repeats=10, preds=True, features=True)
run_random_forest(m_selected, dt, f"{OUTDIR}/{SUBDIR}/mut_", repeats=10, preds=True, features=True)

# Early Fusion Models

In [ ]:
# add unique colnames for each modality
atc_uni_selected = atc_uni_selected.add_suffix("_ATAC")
rna_uni_selected = rna_uni_selected.add_suffix("_RNA")
cnv_uni_selected = cnv_uni_selected.add_suffix("_CNV")
mut_uni_selected = mut_uni_selected.add_suffix("_MUT")

a_selected = a_selected.add_suffix("_ATAC")
r_selected = r_selected.add_suffix("_RNA")
c_selected = c_selected.add_suffix("_CNV")
m_selected = m_selected.add_suffix("_MUT")

### 1 - With omic-specific unimodal selected features

In [ ]:
SUBDIR = "ef-1"

# scale ATAC, RNA, and CNV
continuous_df = pd.concat([atc_uni_selected, rna_uni_selected, cnv_uni_selected], axis=1)
scaler = StandardScaler()
continuous_scaled = pd.DataFrame(scaler.fit_transform(continuous_df),
                                 index=continuous_df.index,
                                 columns=continuous_df.columns)

# combine dataframes for early fusion
X = pd.concat([continuous_scaled, mut_uni_selected], axis=1)

# run models
assert (X.index == dt.index).all(), "sample mismatch"
run_elastic_net(X, dt, f"{OUTDIR}/{SUBDIR}/")
run_random_forest(X, dt, f"{OUTDIR}/{SUBDIR}/")

### 2 - With features selected by >1 omics

In [ ]:
SUBDIR = "ef-2"

# scale ATAC, RNA, and CNV
continuous_df = pd.concat([a_selected, r_selected, c_selected], axis=1)
scaler = StandardScaler()
continuous_scaled = pd.DataFrame(scaler.fit_transform(continuous_df),
                                 index=continuous_df.index,
                                 columns=continuous_df.columns)

# combine dataframes for early fusion
X = pd.concat([continuous_scaled, m_selected], axis=1)

# run models
assert (X.index == dt.index).all(), "sample mismatch"
run_elastic_net(X, dt, f"{OUTDIR}/{SUBDIR}/")
run_random_forest(X, dt, f"{OUTDIR}/{SUBDIR}/")